# Cross-Validating the AST Taxonomy with Semantic Embeddings (Q5, Top-K)

Independent check on the AST-based structural taxonomy from the companion
paper: does a code-embedding model, given **no knowledge of which AST rule
matched a submission**, recover the same manual/builtin/novel split that the
structural competencies (`SelectionStep`, `FindExtremum`) already assign?

**Design:**
1. Label every Q5 submission by its structural match: `manual`
   (`SelectionStep`), `builtin` (`FindExtremum` but not `SelectionStep`), or
   `novel` (matches neither, i.e. `FindExtremum_final == 0`).
2. Split 70% train / 30% test.
3. Build one embedding centroid per label from the training 70%.
4. Classify the held-out 30% by nearest centroid, using embeddings only.
5. Compare the semantic prediction against the real structural label.

**What this tells you either way:**
- High agreement -> real, independent evidence the AST taxonomy's categories
  correspond to genuine distinctions in embedding space, not an artifact of
  the rules themselves.
- Low agreement, especially on `novel` -> a legitimate finding worth
  reporting: either the embedding model isn't separating these cases well,
  or `novel` isn't one coherent strategy and needs sub-clustering rather than
  a single centroid (`novel` is "everything the rules didn't recognize", not
  one implementation approach -- worth checking directly if that row looks
  weak, see the note at the end of this notebook).

Requires: your Drive folder already has `competency.py`, `topk_report.csv`,
and the Refactory Q5 `correct/`/`wrong/` `.py` files (same layout as the
semantic-backup-validation notebook).


## 1. Setup

In [ ]:
!pip install -q transformers torch pandas scikit-learn matplotlib

import torch
print("CUDA available:", torch.cuda.is_available())
print("Device:", "cuda" if torch.cuda.is_available() else "cpu (fine for codet5p-110m-embedding on short snippets)")

import transformers
print("transformers version:", transformers.__version__)


## 2. Mount Google Drive

Same `PROJECT_DIR` convention as the semantic-backup-validation notebook --
adjust the path if your folder is named differently.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

PROJECT_DIR = '/content/drive/MyDrive/competency_extraction'  # <-- adjust if needed

import os
assert os.path.isdir(PROJECT_DIR), f"Not found: {PROJECT_DIR} -- check the folder name/path in Drive"
print("Contents:", os.listdir(PROJECT_DIR))


## 3. Load the embedding model

`Salesforce/codet5p-110m-embedding` -- trained with a contrastive objective
for code similarity (unlike raw `codebert-base`, which collapses under mean
pooling; see the semantic-backup-validation notebook for that finding).

Includes the config patch needed to work around a compatibility bug in the
model's 2023-era remote code (`AttributeError:
'CodeT5pEmbeddingConfig' object has no attribute 'is_decoder'` on recent
`transformers` -- this patch avoids needing any version pin).


In [ ]:
from transformers import AutoConfig, AutoModel, AutoTokenizer
import torch

_checkpoint = "Salesforce/codet5p-110m-embedding"
_device = "cuda" if torch.cuda.is_available() else "cpu"

_tokenizer = AutoTokenizer.from_pretrained(_checkpoint, trust_remote_code=True)

_config = AutoConfig.from_pretrained(_checkpoint, trust_remote_code=True)
for _attr, _default in [("is_decoder", False), ("is_encoder_decoder", True),
                         ("pad_token_id", 0), ("eos_token_id", 2)]:
    if not hasattr(_config, _attr):
        setattr(_config, _attr, _default)

_model = AutoModel.from_pretrained(_checkpoint, config=_config, trust_remote_code=True).to(_device)
_model.eval()

@torch.no_grad()
def codebert_embed(code: str):
    inputs = _tokenizer.encode(code, return_tensors="pt", truncation=True, max_length=512).to(_device)
    embedding = _model(inputs)[0]  # already L2-normalized: norm == 1.0
    return embedding.cpu().numpy()

def cosine_sim(a, b) -> float:
    import numpy as np
    return float(np.dot(a, b))

# Sanity check -- should show a real gap, not near-identical scores
related_a = "def f(lst):\n    best = lst[0]\n    for x in lst:\n        if x > best:\n            best = x\n    return best"
related_b = "def f(lst):\n    return max(lst)"
unrelated  = "def greet(name):\n    return 'Hello, ' + name"
print("related vs related:", cosine_sim(codebert_embed(related_a), codebert_embed(related_b)))
print("related vs unrelated:", cosine_sim(codebert_embed(related_a), codebert_embed(unrelated)))


## 4. Load the batch report and assign structural labels

Reads `topk_report.csv` (from `batch_report.py`'s Q5 run) and labels every
submission `manual`, `builtin`, `novel`, or `partial` based on which
structural competency actually matched.


In [ ]:
import pandas as pd

report = pd.read_csv(f"{PROJECT_DIR}/topk_report.csv")

def structural_label(row, threshold=0.99):
    if row["SelectionStep_final"] >= threshold:
        return "manual"
    elif row["FindExtremum_final"] >= threshold:
        return "builtin"
    elif row["FindExtremum_final"] == 0:
        return "novel"
    else:
        return "partial"  # nonzero but matches neither branch fully -- excluded below

report["structural_label"] = report.apply(structural_label, axis=1)
print(report["structural_label"].value_counts())


## 5. 70/30 split

`SHUFFLE_BEFORE_SPLIT` defaults to `False` -- a literal first-70%/last-30%
split on `topk_report.csv`'s existing row order. Note that
`batch_report.py` builds this CSV via `sorted(glob.glob(...))`, so rows are
in alphabetical filename order -- if `correct_5_*.py` and `wrong_5_*.py`
happen to cluster unevenly under that ordering, the split may be less random
than you'd want. Flip the flag to `True` for a shuffled split instead (fixed
seed, so it's reproducible) if you'd rather rule that out.


In [ ]:
SHUFFLE_BEFORE_SPLIT = False  # flip to True for a randomized, reproducible split instead
SPLIT_SEED = 0

# Keep only the three clean groups this experiment is about
labeled = report[report["structural_label"].isin(["manual", "builtin", "novel"])].reset_index(drop=True)

if SHUFFLE_BEFORE_SPLIT:
    labeled = labeled.sample(frac=1, random_state=SPLIT_SEED).reset_index(drop=True)

split_point = int(len(labeled) * 0.7)
train_df = labeled.iloc[:split_point]
test_df = labeled.iloc[split_point:]

print(f"train: {len(train_df)}  test: {len(test_df)}")
print("\ntrain label counts:\n", train_df["structural_label"].value_counts())
print("\ntest label counts:\n", test_df["structural_label"].value_counts())


## 6. Build one centroid per label from the training 70%


In [ ]:
import os
import numpy as np

FOLDER = f"{PROJECT_DIR}/refactory/data/question_5/code"

def read_src(fname):
    for sub in ["correct", "wrong"]:
        p = f"{FOLDER}/{sub}/{fname}"
        if os.path.exists(p):
            return open(p, encoding="utf-8").read()
    return None

centroids = {}
for label in ["manual", "builtin", "novel"]:
    files = train_df[train_df["structural_label"] == label]["file"].tolist()
    embeddings = [codebert_embed(read_src(f)) for f in files if read_src(f) is not None]
    centroids[label] = np.mean(embeddings, axis=0)
    print(f"{label}: {len(embeddings)} training submissions embedded")


## 7. Classify the held-out 30% by nearest centroid

Embeddings only -- the structural label is used solely to score the
prediction afterward, never as an input to the classification itself.


In [ ]:
rows = []
for _, row in test_df.iterrows():
    src = read_src(row["file"])
    if src is None:
        continue
    emb = codebert_embed(src)
    sims = {label: cosine_sim(emb, c) for label, c in centroids.items()}
    predicted = max(sims, key=sims.get)
    rows.append({
        "file": row["file"],
        "structural_label": row["structural_label"],
        "semantic_predicted": predicted,
        **{f"sim_{k}": v for k, v in sims.items()},
    })

results_df = pd.DataFrame(rows)
results_df.head(10)


## 8. Confusion matrix and agreement


In [ ]:
from sklearn.metrics import confusion_matrix, accuracy_score, classification_report

labels_order = ["manual", "builtin", "novel"]
cm = confusion_matrix(results_df["structural_label"], results_df["semantic_predicted"], labels=labels_order)

print("confusion matrix [rows=structural label, cols=semantic prediction]")
print(pd.DataFrame(cm, index=labels_order, columns=labels_order))

print("\noverall accuracy:", accuracy_score(results_df["structural_label"], results_df["semantic_predicted"]))
print("\n", classification_report(results_df["structural_label"], results_df["semantic_predicted"], labels=labels_order))


## 9. If the `novel` row looks weak: check whether `novel` is even one cluster

`novel` means "the rules didn't recognize this" -- it isn't guaranteed to be
one coherent implementation strategy. This cell shows how tightly (or
loosely) the real `novel` submissions actually cluster around their own
centroid, and which specific files are the closest/furthest from it -- useful
for deciding whether `novel` needs to be split into sub-strategies rather
than treated as one group.


In [ ]:
novel_test = results_df[results_df["structural_label"] == "novel"].copy()
novel_test = novel_test.sort_values("sim_novel", ascending=False)

print(f"novel test submissions: {len(novel_test)}")
print(f"sim_novel stats: mean={novel_test['sim_novel'].mean():.3f} std={novel_test['sim_novel'].std():.3f}")
print("\nclosest to the novel centroid (most 'typical' novel submissions):")
print(novel_test[["file", "sim_novel", "sim_manual", "sim_builtin"]].head(5))
print("\nfurthest from the novel centroid (most atypical -- possibly a distinct sub-strategy):")
print(novel_test[["file", "sim_novel", "sim_manual", "sim_builtin"]].tail(5))


## 10. Save results


In [ ]:
results_df.to_csv(f"{PROJECT_DIR}/ast_taxonomy_crossvalidation_results.csv", index=False)

from google.colab import files
files.download(f"{PROJECT_DIR}/ast_taxonomy_crossvalidation_results.csv")
